In [3]:
import requests
import pandas as pd

url = "https://aviationweather.gov/api/data/metar"
params = {
    "ids": "CYYZ,CYVR,CYUL,CYEG,CYYC,CYOW,CYWG,CYQB",
    "format": "json",
    "hours": 15
}
response = requests.get(url, params = params)
data = response.json()

df = pd.DataFrame(data)
print(df.shape)
df.head()

(143, 23)


,icaoId,receiptTime,obsTime,reportTime,temp,dewp,wdir,wspd,visib,altim,...,rawOb,lat,lon,elev,name,cover,clouds,fltCat,wgst,wxString
0,CYYC,2026-06-30T20:08:11.785Z,1782849600,2026-06-30T20:00:00.000Z,19,13,40,8,8,1008.5,...,METAR CYYC 302000Z 04008KT 8SM SCT031 BKN098 B...,51.116,-114.011,1085,"Calgary Intl, AB, CA",BKN,"[{'cover': 'SCT', 'base': 3100}, {'cover': 'BK...",VFR,NaN,NaN
1,CYWG,2026-06-30T20:08:11.707Z,1782849600,2026-06-30T20:00:00.000Z,25,15,230,26,15,997.4,...,METAR CYWG 302000Z 23026G38KT 15SM FEW054 FEW2...,49.916,-97.249,238,"Winnipeg Intl Arpt, MB, CA",FEW,"[{'cover': 'FEW', 'base': 5400}, {'cover': 'FE...",VFR,38.0,NaN
2,CYQB,2026-06-30T20:08:17.557Z,1782849600,2026-06-30T20:00:00.000Z,25,18,260,5,30,1014.0,...,METAR CYQB 302000Z CCA 26005KT 30SM BKN170 OVC...,46.791,-71.396,73,"Quebec/Lesage Intl, QC, CA",OVC,"[{'cover': 'BKN', 'base': 17000}, {'cover': 'O...",VFR,NaN,NaN
3,CYVR,2026-06-30T20:08:11.691Z,1782849600,2026-06-30T20:00:00.000Z,18,12,280,10,30,1016.7,...,METAR CYVR 302000Z 28010G15KT 30SM FEW068 BKN0...,49.183,-123.168,2,"Vancouver Intl, BC, CA",BKN,"[{'cover': 'FEW', 'base': 6800}, {'cover': 'BK...",VFR,15.0,NaN
4,CYUL,2026-06-30T20:08:11.641Z,1782849600,2026-06-30T20:00:00.000Z,27,17,180,8,15,1013.3,...,METAR CYUL 302000Z 18008KT 15SM BKN120 BKN160 ...,45.468,-73.742,31,"Montreal/Trudeau Intl, QC, CA",OVC,"[{'cover': 'BKN', 'base': 12000}, {'cover': 'B...",VFR,NaN,NaN


In [4]:
def get_ceiling(clouds):
    ceiling_layers = [c['base'] for c in clouds if c['cover'] in ('BKN', 'OVC')]
    return min(ceiling_layers) if ceiling_layers else None

df['ceiling'] = df['clouds'].apply(get_ceiling)

In [8]:
df['temp_dewp_spread'] = df['temp'] - df['dewp']
df['fltCat'].value_counts()

fltCat
VFR     132
MVFR      8
IFR       3
Name: count, dtype: int64

In [16]:
df['visib'] = df['visib'].astype(str).str.replace('+', '', regex=False).astype(float)

df['fltCat_binary'] = df['fltCat'].apply(lambda x: 'VFR' if x == 'VFR' else 'non-VFR')
df['fltCat_binary'].value_counts()

                                        

fltCat_binary
VFR        132
non-VFR     11
Name: count, dtype: int64

In [17]:
X = df[['visib', 'wspd', 'temp_dewp_spread', 'ceiling']]
y = df['fltCat_binary']

In [20]:
import joblib 
model = joblib.load('metar_classifier.pkl')

predictions = model.predict(X)
comparison = pd.DataFrame({
    'real': y,
    'predicted': predictions
})
print(comparison)

        real predicted
0        VFR       VFR
1        VFR       VFR
2        VFR       VFR
3        VFR       VFR
4        VFR       VFR
..       ...       ...
138      VFR       VFR
139      VFR       VFR
140  non-VFR   non-VFR
141      VFR       VFR
142      VFR       VFR

[143 rows x 2 columns]


In [21]:
from sklearn.metrics import accuracy_score
print(accuracy_score(y, predictions))

1.0
